# replace-final-head — worked example 3: Count Total Parameters Before and After Replacing the Head

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `replace-final-head`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When you replace a pretrained head (e.g., 1000-class) with a smaller head (e.g., 10-class), the total parameter count decreases. The backbone parameters are unchanged. The old head's `in_features * 1000 + 1000` parameters are replaced by `in_features * new_classes + new_classes`. Counting parameters before and after the swap confirms the replacement is complete and nothing is duplicated.

## Worked solution

**Step 1 — count parameters.** `sum(p.numel() for p in model.parameters())` gives the total parameter count including all layers.

**Step 2 — record the before count.** Before any swap, store the parameter count and the specific count for `model.fc`.

**Step 3 — compute expected change.** The difference should be exactly `(old_out - new_out) * in_features + (old_out - new_out)` = `(old_out - new_out) * (in_features + 1)`.

**Step 4 — perform the swap.** Read `in_features`, assign the new `nn.Linear`.

**Step 5 — verify the count delta.** The new total should equal the old total minus the exact expected reduction.

In [ ]:
import torch as t
import torch.nn as nn

class ToyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(nn.Linear(8, 32), nn.ReLU(), nn.Linear(32, 16))
        self.fc = nn.Linear(16, 100)  # 100-class original
    def forward(self, x):
        return self.fc(self.backbone(x))

def count_params(model): 
    return sum(p.numel() for p in model.parameters())

# --- exercise and print ---
t.manual_seed(0)
model = ToyNet()
new_num_classes = 5

params_before = count_params(model)
old_head_params = sum(p.numel() for p in model.fc.parameters())
print(f'Total params before: {params_before}')
print(f'Old head params:     {old_head_params}  (16*100 + 100 = {16*100+100})')

# Perform the swap
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, new_num_classes)

params_after = count_params(model)
new_head_params = sum(p.numel() for p in model.fc.parameters())
print(f'Total params after:  {params_after}')
print(f'New head params:     {new_head_params}  (16*5 + 5 = {16*5+5})')
print(f'Reduction:           {params_before - params_after}  (expected {old_head_params - new_head_params})')
print(f'Match: {params_before - params_after == old_head_params - new_head_params}')